Setup: Data Paths

Run this notebook first, especially if you're new to the project or on a different
machine than whoever last ran it.

Every other notebook gets its data paths from `src/paths.py`, which defaults to
`<repo>/data/raw` and `<repo>/data/processed`. If your copy of the data lives somewhere
else (a different drive, a shared folder, ...), you don't need to edit every notebook -
just create one small override file, `src/local_paths.py` (gitignored, so it stays local
to your machine and never gets committed):

1. Copy `src/local_paths.example.py` to `src/local_paths.py`
2. Edit `RAW` and `PROCESSED` in that file to point at your actual data folders
3. Re-run the cells below to confirm everything is found

If your data is already at `<repo>/data/raw`, skip that - the defaults just work.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import RAW, PROCESSED, RESULTS

local_paths_file = Path.cwd().parent / "src" / "local_paths.py"

print("Using local_paths.py override:", local_paths_file.exists())
print("RAW       =", RAW)
print("PROCESSED =", PROCESSED)
print("RESULTS   =", RESULTS)

Check That the Expected Raw Data Is Actually There

One entry per dataset used somewhere in the pipeline (notebooks 01-08). A missing file
here is exactly what would otherwise surface as a confusing `FileNotFoundError` several
notebooks deep.

In [ ]:
expected_raw_files = [
    "gadm/gadm_410-levels-ADM_1-AUT.gpkg",
    "wdpa/WDPA_Oct2022_Public_shp-AUT.tif",
    "copernicus-glc/PROBAV_LC100_global_v3.0.1_2019-nrt_Discrete-Classification-map_EPSG-4326-AT.tif",
    "gebco/GEBCO_2014_2D-AT.nc",
    "ne_10m_airports.gpkg",
    "ne_10m_roads.gpkg",
    "AU-2019.nc",
    "global-power-plant-database/global_power_plant_database.csv",
    "gegis/load.csv",
]

missing = []
for rel_path in expected_raw_files:
    full_path = RAW / rel_path
    found = full_path.exists()
    print(("OK   " if found else "MISSING"), rel_path)
    if not found:
        missing.append(rel_path)

if missing:
    print(f"\n{len(missing)} file(s) missing under {RAW} - fix RAW in src/local_paths.py "
          "(see markdown above) or place the data there, then re-run this cell.")
else:
    print("\nAll expected raw data found.")

Check That PROCESSED and RESULTS Are Writable

These get created automatically by `src/paths.py`, but confirming here catches
permission issues (e.g. a read-only shared drive) before they surface mid-notebook.

In [ ]:
for folder in [PROCESSED, RESULTS]:
    probe = folder / ".write_test"
    try:
        probe.write_text("ok")
        probe.unlink()
        print("OK, writable:", folder)
    except OSError as e:
        print("NOT writable:", folder, "-", e)